In [1]:

# 2) Install/upgrade the required Python libraries.
# - bitsandbytes enables 4-bit/8-bit quantized loading (saves VRAM).
# - sentencepiece is required for Llama tokenization.
# - accelerate helps with device placement and offloading.
# If any installs fail in your environment, re-run this cell after adjusting your CUDA/PyTorch setup.
%pip install -q -U transformers accelerate huggingface_hub sentencepiece bitsandbytes


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
#https://huggingface.co/bigscience/bloomz-7b1-mt MEJOR MODELO
#Nuevo modelo Qwen3.0

bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token_id = tok.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_4bit,
    device_map={"": 0},          # todo en GPU (6 GB lo aguanta bien)
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

# ids útiles para stop
EOS_ID = tok.eos_token_id
IM_END_ID = tok.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [i for i in [EOS_ID, IM_END_ID] if i is not None]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
from transformers import StoppingCriteria, StoppingCriteriaList
from transformers.utils import logging
logging.set_verbosity_error()
# o:
import os; os.environ["TRANSFORMERS_VERBOSITY"] = "error"

class StopOnAnyId(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set(stop_ids)
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -1].item() in self.stop_ids

def translate_es_to_en(text: str, max_new_tokens: int = 128) -> str:
    system = (
        "You are a professional translator. Translate Spanish to natural, fluent English. "
        "Use idiomatic meaning (not literal). Preserve names and numbers. Output ONLY the translation."
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": text}
    ]

    # prompt del modelo (chat template Qwen)
    input_ids = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    gen_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,                 # greedy determinista
        use_cache=False,                 # ahorra VRAM
        eos_token_id=STOP_IDS,           # parar en <|im_end|> y/o eos
        pad_token_id=tok.pad_token_id,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        stopping_criteria=StoppingCriteriaList([StopOnAnyId(STOP_IDS)]),
        return_dict_in_generate=False,
    )

    new_tokens = gen_ids[0, input_ids.shape[1]:]
    out = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # Limpieza defensiva: si se coló un prefijo o líneas extra, coge la primera “línea útil”
    out = out.split("\n")[0].strip().strip('"')
    return out


In [4]:
# ==== Traducir SRT ES->EN imprimiendo progreso línea a línea ====
import re, io, os, sys, torch

PRINT_LINES = True   # pon a False si no quieres ver cada línea

def parse_srt(srt_text: str):
    srt_text = srt_text.replace("\r\n", "\n").replace("\r", "\n").strip()
    blocks = re.split(r'\n{2,}', srt_text)
    entries = []
    for block in blocks:
        lines = block.strip().split("\n")
        if len(lines) >= 3:
            entries.append({
                "index": lines[0].strip(),
                "times": lines[1].strip(),
                "text":  "\n".join(lines[2:]).strip()
            })
    return entries

def write_srt(entries):
    out = []
    for i, e in enumerate(entries, start=1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip() + "\n"

def _print_line(i, n, j, m, es, en):
    es_short = es if len(es) <= 80 else es[:77] + "..."
    en_short = en if len(en) <= 80 else en[:77] + "..."
    print(f"[{i}/{n}] línea {j}/{m}\n  ES: {es_short}\n  EN: {en_short}\n", flush=True)

# ---- Rutas ----
srt_in  = "sub_es.srt"   # <-- tu SRT en español
srt_out = "sub_en.srt"                  # <-- salida en inglés

if os.path.exists(srt_in):
    with io.open(srt_in, "r", encoding="utf-8") as f:
        srt_text = f.read()

    entries = parse_srt(srt_text)
    total_blocks = len(entries)
    print(f"Found {total_blocks} subtitle blocks. Translating...\n")

    model.eval()
    new_entries = []
    with torch.no_grad():
        for i, e in enumerate(entries, start=1):
            src_block = e["text"]
            lines = (src_block.split("\n") if src_block else [""])
            translated_lines = []
            for j, line in enumerate(lines, start=1):
                stripped = line.strip()
                if stripped:
                    try:
                        en = translate_es_to_en(stripped, max_new_tokens=80)
                    except Exception as ex:
                        en = stripped  # fallback para no romper el SRT
                    translated_lines.append(en)
                    if PRINT_LINES:
                        _print_line(i, total_blocks, j, len(lines), stripped, en)
                else:
                    translated_lines.append("")
                    if PRINT_LINES:
                        _print_line(i, total_blocks, j, len(lines), "", "")
            new_entries.append({"times": e["times"], "text": "\n".join(translated_lines)})

            # Marca de progreso por bloque
            print(f"  ✓ bloque {i}/{total_blocks}\n", flush=True)

    out_text = write_srt(new_entries)

In [ ]:
# ==== ES->EN: crear `new_entries` desde ./srts/sub_es.srt ====
%pip install -q transformers sentencepiece

import os, io, re, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

SRT_IN = os.path.join("srts", "sub_es.srt")   # tu SRT en español
SAVE_EN = True                                 # si quieres guardar sub_en.srt además de crear new_entries

# >>> Añadido: asegurar carpeta de salida
CARPETA_SRTS = os.path.join(os.getcwd(), "srts")
os.makedirs(CARPETA_SRTS, exist_ok=True)

def is_meta_tag(line: str) -> bool:
    return bool(re.fullmatch(r"\[[^\]]+\]", line.strip()))

def parse_srt(text: str):
    blocks = [b for b in text.split("\n\n") if b.strip()]
    out = []
    for b in blocks:
        lines = [l for l in b.splitlines() if l.strip() != "" or True]
        if len(lines) < 2: 
            continue
        idx = lines[0].strip()
        times = lines[1].strip()
        txt = "\n".join(lines[2:]).strip() if len(lines) > 2 else ""
        out.append({"times": times, "text": txt})
    return out

def write_srt(entries):
    out=[]
    for i,e in enumerate(entries,1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip()+"\n"

# 1) Cargar SRT español
with io.open(SRT_IN, "r", encoding="utf-8") as f:
    es_text = f.read()
entries_es = parse_srt(es_text)

# 2) Traducir ES->EN por lotes
model_name = "Helsinki-NLP/opus-mt-es-en"
device = 0 if torch.cuda.is_available() else -1
tok = AutoTokenizer.from_pretrained(model_name)
mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name)
translator = pipeline("translation", model=mdl, tokenizer=tok, device=device)

entries_en = []
BATCH = 16
for e in entries_es:
    src_lines = e["text"].split("\n") if e["text"] else [""]
    tgt_lines = [""] * len(src_lines)
    idxs, batch = [], []
    for j, line in enumerate(src_lines):
        s = line.strip()
        if s and not is_meta_tag(s):
            idxs.append(j); batch.append(s)
    if batch:
        outs = translator(batch, max_length=512, batch_size=BATCH)
        outs_text = [o["translation_text"] for o in outs]
        for k, j in enumerate(idxs):
            tgt_lines[j] = outs_text[k]
    for j, line in enumerate(src_lines):
        if line.strip()=="" or is_meta_tag(line):
            tgt_lines[j] = line
    entries_en.append({"times": e["times"], "text": "\n".join(tgt_lines)})

# 3) Dejar disponible para el selector
new_entries = entries_en  # <<< lo que necesita tu celda de widgets

# 4) Guardar en ./srts/sub_en.srt
if SAVE_EN:
    out_path = os.path.join(CARPETA_SRTS, "sub_en.srt")
    with io.open(out_path, "w", encoding="utf-8") as f:
        f.write(write_srt(entries_en))
    print(f"Creado `new_entries` y guardado: {out_path}")
else:
    print("Creado `new_entries` (sin guardar SRT)")


Note: you may need to restart the kernel to use updated packages.
✅ Creado `new_entries` y guardado: c:\Users\carlos.basallote\Desktop\TFM2\TFM\TFM\code\srts\sub_en.srt


In [ ]:
# ==== Selector de idioma y guardado de SRT (desde `new_entries`) ====
import os, io, re, torch
import ipywidgets as w
from IPython.display import display, clear_output, FileLink
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Carpeta de salida (siempre ./srts)
CARPETA_SRTS = os.path.join(os.getcwd(), "srts")
os.makedirs(CARPETA_SRTS, exist_ok=True)

# Modelos EN->XX
EN_TO_MODEL = {
    "de": "Helsinki-NLP/opus-mt-en-de",   # Alemán
    "fr": "Helsinki-NLP/opus-mt-en-fr",   # Francés
    "it": "Helsinki-NLP/opus-mt-en-it",   # Italiano
    "pt": "Helsinki-NLP/opus-mt-tc-big-en-pt",   # Portugués
    "nl": "Helsinki-NLP/opus-mt-en-nl",   # Neerlandés
    "sv": "Helsinki-NLP/opus-mt-en-sv",   # Sueco
    "da": "Helsinki-NLP/opus-mt-en-da",   # Danés
    "cs": "Helsinki-NLP/opus-mt-en-cs",   # Checo
    "pl": "gsarti/opus-mt-tc-en-pl",      # Polaco
    "uk": "Helsinki-NLP/opus-mt-en-uk",   # Ucraniano
    "el": "Helsinki-NLP/opus-mt-en-el",   # Griego
    "he": "Helsinki-NLP/opus-mt-en-he",   # Hebreo
    "tr": "Helsinki-NLP/opus-mt-tc-big-en-tr",   # Turco
    "ro": "Helsinki-NLP/opus-mt-en-ro",   # Rumano
    "zh-hans": "Helsinki-NLP/opus-mt-en-zh",     # Chino simplificado
    "ar": "Helsinki-NLP/opus-mt-en-ar",   # Árabe
    "ru": "Helsinki-NLP/opus-mt-en-ru",   # Ruso
    "ja": "Helsinki-NLP/opus-mt-en-ja",   # Japonés
    "ko": "Helsinki-NLP/opus-mt-tc-big-en-ko"    # Coreano
}

# Nombres bonitos
LANG_LABELS = {
    "en": "Inglés",
    "de": "Alemán",
    "fr": "Francés",
    "it": "Italiano",
    "pt": "Portugués",
    "nl": "Neerlandés",
    "sv": "Sueco",
    "da": "Danés",
    "cs": "Checo",
    "pl": "Polaco",
    "uk": "Ucraniano",
    "el": "Griego",
    "he": "Hebreo",
    "tr": "Turco",
    "ro": "Rumano",
    "zh-hans": "Chino (simplificado)",
    "ar": "Árabe",
    "ru": "Ruso",
    "ja": "Japonés",
    "ko": "Coreano",
}

def label(code: str) -> str:
    return LANG_LABELS.get(code, code)

# -------- Helpers SRT --------
def write_srt(entries):
    out=[]
    for i,e in enumerate(entries,1):
        out += [str(i), e["times"], e["text"], ""]
    return "\n".join(out).rstrip()+"\n"

def is_meta_tag(line: str) -> bool:
    return bool(re.fullmatch(r"\[[^\]]+\]", line.strip()))

def translate_srt_from_en(entries_en, target_lang, batch_size=16, print_lines=True):
    if target_lang == "en":
        return entries_en

    model_name = EN_TO_MODEL[target_lang]
    device = 0 if torch.cuda.is_available() else -1
    tok_mt = AutoTokenizer.from_pretrained(model_name)
    mdl_mt = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    translator = pipeline("translation", model=mdl_mt, tokenizer=tok_mt, device=device)

    total = len(entries_en)
    out_entries = []
    for i, e in enumerate(entries_en, start=1):
        src_lines = e["text"].split("\n") if e["text"] else [""]
        tgt_lines = [""] * len(src_lines)

        idxs, batch = [], []
        for j, line in enumerate(src_lines):
            s = line.strip()
            if s and not is_meta_tag(s):
                idxs.append(j); batch.append(s)

        if batch:
            outs = translator(batch, max_length=512, batch_size=batch_size)
            outs_text = [o["translation_text"] for o in outs]
            for k, j in enumerate(idxs):
                tgt_lines[j] = outs_text[k]
                if print_lines:
                    es = src_lines[j]
                    en = tgt_lines[j]
                    es_s = es if len(es)<=80 else es[:77]+"..."
                    en_s = en if len(en)<=80 else en[:77]+"..."
                    print(f"[{i}/{total}] línea {j+1}/{len(src_lines)}\n  EN: {es_s}\n  {target_lang.upper()}: {en_s}\n")

        for j, line in enumerate(src_lines):
            if line.strip()=="" or is_meta_tag(line):
                tgt_lines[j] = line

        out_entries.append({"times": e["times"], "text": "\n".join(tgt_lines)})

        if i % 20 == 0 or i == total:
            print(f"  ✓ bloque {i}/{total}\n", flush=True)
    return out_entries

# -------- Widgets --------
if "new_entries" not in globals() or not isinstance(new_entries, list) or not new_entries:
    raise RuntimeError("`new_entries` no está definido o está vacío. Ejecuta antes la celda ES->EN que lo crea.")

lang_options = [(label("en"), "en")] + [(label(k), k) for k in EN_TO_MODEL.keys()]

dd_lang = w.Dropdown(options=lang_options, value="de", description="Destino:")
cb_print = w.Checkbox(value=True, description="Mostrar líneas")
bs = w.IntSlider(value=16, min=4, max=64, step=4, description="Batch")
btn = w.Button(description="Traducir y guardar", button_style="primary")
out = w.Output()

def on_click(_):
    with out:
        clear_output(wait=True)
        target = dd_lang.value
        nombre = LANG_LABELS.get(target, target)
        print(f"Destino: {nombre}\n")

        if target == "en":
            srt_path = os.path.join(CARPETA_SRTS, "sub_en.srt")
            with io.open(srt_path, "w", encoding="utf-8") as f:
                f.write(write_srt(new_entries))
            print(f"Guardado SRT {nombre}: {srt_path}")
            display(FileLink(srt_path))

            # Mantener solo sub_es.srt y sub_en.srt
            for fname in os.listdir(CARPETA_SRTS):
                if fname.endswith(".srt") and fname not in ("sub_es.srt", "sub_en.srt"):
                    os.remove(os.path.join(CARPETA_SRTS, fname))
                    print(f"Eliminado: {fname}")
            return

        if target not in EN_TO_MODEL:
            print(f"Idioma '{target}' no soportado en EN_TO_MODEL.")
            return

        xx_entries = translate_srt_from_en(new_entries, target, batch_size=bs.value, print_lines=cb_print.value)
        srt_path = os.path.join(CARPETA_SRTS, f"sub_{target}.srt")
        with io.open(srt_path, "w", encoding="utf-8") as f:
            f.write(write_srt(xx_entries))
        print(f"Guardado SRT {nombre}: {srt_path}")
        display(FileLink(srt_path))

        # Mantener solo sub_es.srt y el idioma elegido
        keep_files = {f"sub_es.srt", f"sub_{target}.srt"}
        for fname in os.listdir(CARPETA_SRTS):
            if fname.endswith(".srt") and fname not in keep_files:
                os.remove(os.path.join(CARPETA_SRTS, fname))
                print(f"Eliminado: {fname}")

btn.on_click(on_click)
display(w.HBox([dd_lang, cb_print, bs, btn]), out)

Output()